In [ ]:
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.model_selection import KFold
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor
from torchvision import transforms
import torchvision
from sklearn.metrics import (f1_score, confusion_matrix)
import csv
import numpy as np
from ptflops import get_model_complexity_info
import time
import os 

In [ ]:
#fixed seed 
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
np.random.seed(seed)


In [ ]:
def seed_worker(worker_id):
    """Ensures reproducibility across data loading workers."""
    np.random.seed(seed + worker_id)
    random.seed(seed + worker_id)

g = torch.Generator()
g.manual_seed(seed)

In [ ]:
valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
# Define the dataset and transformations
train_dataset = ImageFolder(root="/home/gpu1/Desktop/rut_ws/tom_classification_kfold+test/train", transform=valid_transform)
test_dataset = ImageFolder(root="/home/gpu1/Desktop/rut_ws/tom_classification_kfold+test/test", transform=valid_transform)
print("Classes: ")
class_names = train_dataset.classes
print(train_dataset.classes)

In [ ]:
class TransformDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        img, label = self.dataset[actual_idx]
        if self.transform:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.indices)

In [ ]:
# Define the training function
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    loss_train = 0
    class_weights = torch.tensor([1, 1.478787879, 1.33820841, 1.5, 1.095808383], dtype=torch.float32)
    class_weights = class_weights.to(device)
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

In [ ]:
def evaluate(model, device, valid_loader, epoch, num_samples, save_dir, fold, epoch_val_loss, epoch_val_acc, epochs_f1_score, count_not_improve, max_patience):
    
    model.eval()
    test_loss = 0
    correct = 0
    all_preds = []
    all_targets = []

    criterion = nn.CrossEntropyLoss(reduction="sum")
    
    with torch.no_grad():
        for data, target in valid_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            all_preds.extend(pred.cpu().numpy().flatten())  
            all_targets.extend(target.cpu().numpy().flatten())  

    test_loss /= num_samples
    accuracy = 100.0 * correct / num_samples
    f1 = f1_score(all_targets, all_preds, average="weighted")
    epoch_val_loss.append(test_loss)
    epoch_val_acc.append(accuracy)
    epochs_f1_score.append(f1)

    return all_targets, all_preds
 
    #plot Conf metrics
    #plt.savefig(save_path, dpi=300)
    #plt.close()

In [ ]:
import matplotlib.pyplot as plt
import os
import seaborn as sns

# Define the number of folds and batch size
k_folds = 10
batch_size = 96
epochs = 1000
drop_last = False
lr0 = 0.001

# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the k-fold cross validation
kf = KFold(n_splits=k_folds, shuffle=True, random_state=seed)
print(f"Number of splits: {kf.get_n_splits()}")